# Run baseline (smoke test)
このノートブックはサンプルデータを使って `competitions/rna2/src/baseline` のコアロジックを簡単に検証します。

In [2]:
import sys, os
# baseline パッケージをインポートできるようにパスを追加
# ノートブック実行時の作業ディレクトリはノートブックの場所になるため、
# そこから見た `../src` を追加してリポジトリ内の baseline をインポート可能にする。
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from baseline.data import load_sequences, load_labels
from baseline.template_model import TemplateRepository
from baseline.search import seq_identity
from baseline.predict import generate_submission
# データディレクトリはノートブックから見た相対パスで指定
data_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'stanford-rna-3d-folding-2'))
print('data_dir ->', data_dir)
test_seq = load_sequences(os.path.join(data_dir, 'test_sequences.csv'))
train_seq = load_sequences(os.path.join(data_dir, 'train_sequences.csv'))
train_labels = load_labels(os.path.join(data_dir, 'train_labels.csv'))
print('loaded:', test_seq.shape, train_seq.shape, train_labels.shape)


data_dir -> /Users/tatsuki/work/kaggle/kauto/competitions/rna2/data/stanford-rna-3d-folding-2


/Users/tatsuki/work/kaggle/kauto/competitions/rna2/src/baseline/data.py:37: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


loaded: (28, 2) (5716, 2) (7794971, 9)


In [3]:
# TemplateRepository の学習と最良テンプレート探索、簡易出力生成
repo = TemplateRepository()
repo.fit(train_seq, train_labels)
# テンプレートが作成されているか簡単に確認
print('templates count:', len(repo.templates))
sub = generate_submission(test_seq, repo, seq_identity, n_structures=2)
print('submission shape:', sub.shape)
sub.head()

templates count: 5716
submission shape: (9563, 9)


,ID,resname,resid,x_1,y_1,z_1,chain,copy,target_id
0,8ZNQ_1,G,1,-3.057,-19.901,4.449,A,1,1HWQ
1,8ZNQ_2,G,2,-1.417,-17.912,9.068,A,1,1HWQ
2,8ZNQ_3,U,3,1.834,-14.640,11.569,A,1,1HWQ
3,8ZNQ_4,G,4,5.265,-10.734,11.693,A,1,1HWQ
4,8ZNQ_5,C,5,7.173,-7.044,8.684,A,1,1HWQ


In [ ]:
# 結果を保存
out_path = os.path.abspath(os.path.join(os.getcwd(), 'smoke_submission.csv'))
sub.to_csv(out_path, index=False)
print('saved ->', out_path)
